# 01 — Data Cleaning
### E-Commerce Customer Analytics Dashboard

**Objective:** Load the raw e-commerce export, identify data quality issues, and
produce a clean, analysis-ready dataset (`ecommerce_clean.csv`) for the rest of
the project (feature engineering, EDA, RFM, SQL, Power BI).

**Steps covered:**
1. Load & inspect raw data
2. Handle missing values
3. Remove duplicates
4. Convert / standardize date columns
5. Standardize text fields (case, whitespace)
6. Validate numerical columns (negative/zero values, outliers)
7. Save cleaned dataset


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

RAW_PATH = '../data/ecommerce_raw.csv'
CLEAN_PATH = '../data/ecommerce_clean.csv'

df = pd.read_csv(RAW_PATH)
print(f"Shape: {df.shape}")
df.head()


Shape: (12180, 17)


,Order_ID,Order_Date,Customer_ID,Customer_Name,Gender,Age,City,State,Region,Product_Category,Product_Name,Quantity,Unit_Price,Discount,Sales,Profit,Payment_Method
0,ORD010477,2024-09-08,CUST01027,Aarav Bava,Female,34.0,Nagpur,Maharashtra,West,Beauty & Personal Care,Perfume,1,1366.44,0.05,1298.12,614.90,Wallet
1,ORD008646,05-Oct-2023,CUST00581,Bhavya Borah,Male,37.0,Ranchi,Jharkhand,East,Sports & Fitness,Cricket Bat,2,5190.40,0.00,10380.80,4152.32,Debit Card
2,ORD011576,2023-08-08,CUST00653,Sanya Wable,Female,31.0,Bhubaneswar,Odisha,East,Books,Self-Help Book,1,722.57,0.00,722.57,289.03,Wallet
3,ORD003500,07-Nov-2022,CUST00342,Teerth Sawhney,Female,36.0,Jaipur,Rajasthan,North,Groceries,Dry Fruits Combo,1,345.45,0.00,345.45,96.73,Debit Card
4,ORD008417,2023-10-03,CUST00588,Gauri Som,Female,25.0,Chennai,Tamil Nadu,South,Groceries,Green Tea Pack,1,346.70,NaN,329.36,79.74,Cash on Delivery


## 1. Initial Inspection

In [2]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 12180 entries, 0 to 12179
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          12180 non-null  str    
 1   Order_Date        12180 non-null  str    
 2   Customer_ID       12180 non-null  str    
 3   Customer_Name     12058 non-null  str    
 4   Gender            12180 non-null  str    
 5   Age               11693 non-null  float64
 6   City              12119 non-null  str    
 7   State             12180 non-null  str    
 8   Region            12180 non-null  str    
 9   Product_Category  12180 non-null  str    
 10  Product_Name      12180 non-null  str    
 11  Quantity          12180 non-null  int64  
 12  Unit_Price        12180 non-null  float64
 13  Discount          11815 non-null  float64
 14  Sales             12180 non-null  float64
 15  Profit            12180 non-null  float64
 16  Payment_Method    12180 non-null  str    
dtypes: f

In [3]:
print("Missing values per column:")
print(df.isna().sum())
print()
print("Missing % per column:")
print((df.isna().mean() * 100).round(2))


Missing values per column:
Order_ID              0
Order_Date            0
Customer_ID           0
Customer_Name       122
Gender                0
Age                 487
City                 61
State                 0
Region                0
Product_Category      0
Product_Name          0
Quantity              0
Unit_Price            0
Discount            365
Sales                 0
Profit                0
Payment_Method        0
dtype: int64

Missing % per column:
Order_ID            0.0
Order_Date          0.0
Customer_ID         0.0
Customer_Name       1.0
Gender              0.0
Age                 4.0
City                0.5
State               0.0
Region              0.0
Product_Category    0.0
Product_Name        0.0
Quantity            0.0
Unit_Price          0.0
Discount            3.0
Sales               0.0
Profit              0.0
Payment_Method      0.0
dtype: float64


In [4]:
print("Duplicate rows (exact match):", df.duplicated().sum())
print("Duplicate Order_IDs:", df['Order_ID'].duplicated().sum())


Duplicate rows (exact match): 38
Duplicate Order_IDs: 180


## 2. Remove Duplicates

We drop fully duplicated rows. We keep the first occurrence, since duplicate
`Order_ID`s with identical data are export artifacts, not separate orders.


In [5]:
before = len(df)
df = df.drop_duplicates(keep='first').reset_index(drop=True)
after = len(df)
print(f"Removed {before - after} duplicate rows. New shape: {df.shape}")


Removed 38 duplicate rows. New shape: (12142, 17)


## 3. Standardize Date Columns

The raw export mixes three date formats (`YYYY-MM-DD`, `DD/MM/YYYY`, `DD-Mon-YYYY`).
We parse all of them into a single `datetime64` column.


In [6]:
def parse_mixed_date(value):
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%d-%b-%Y'):
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df['Order_Date'] = df['Order_Date'].apply(parse_mixed_date)

print("Unparsed dates after conversion:", df['Order_Date'].isna().sum())
print("Date range:", df['Order_Date'].min(), "to", df['Order_Date'].max())
df[['Order_ID', 'Order_Date']].head()


Unparsed dates after conversion: 0
Date range: 2022-01-01 00:00:00 to 2024-12-28 00:00:00


,Order_ID,Order_Date
0,ORD010477,2024-09-08
1,ORD008646,2023-10-05
2,ORD011576,2023-08-08
3,ORD003500,2022-11-07
4,ORD008417,2023-10-03


## 4. Standardize Text Fields

Trim whitespace and apply consistent casing (Title Case for names/locations,
fixed category labels for Gender and Payment_Method) so groupings in SQL/
Power BI don't fragment (e.g. `"mumbai"`, `"MUMBAI "`, `"Mumbai"` all becoming
one value).


In [7]:
text_cols = ['Customer_Name', 'Gender', 'City', 'State', 'Region',
             'Product_Category', 'Product_Name', 'Payment_Method']

for col in text_cols:
    df[col] = df[col].astype('string').str.strip()

# Title-case location & name fields
for col in ['Customer_Name', 'City', 'State', 'Region', 'Product_Category', 'Product_Name']:
    df[col] = df[col].str.title()

# Standardize Gender to a fixed set
df['Gender'] = df['Gender'].str.capitalize()
df['Gender'] = df['Gender'].replace({'M': 'Male', 'F': 'Female'})

# Standardize Payment_Method casing (title case, but keep acronyms like UPI)
df['Payment_Method'] = df['Payment_Method'].str.title()
df['Payment_Method'] = df['Payment_Method'].replace({'Upi': 'UPI'})

print(df['Gender'].value_counts())
print()
print(df['Payment_Method'].value_counts())


Gender
Female    6431
Male      5711
Name: count, dtype: Int64

Payment_Method
Wallet              2086
Net Banking         2058
Credit Card         2016
Cash On Delivery    1998
Debit Card          1995
UPI                 1989
Name: count, dtype: Int64


In [8]:
print("Unique cities sample:", sorted(df['City'].dropna().unique())[:10])
print("Total unique cities:", df['City'].nunique())


Unique cities sample: ['Ahmedabad', 'Bengaluru', 'Bhubaneswar', 'Chandigarh', 'Chennai', 'Coimbatore', 'Dehradun', 'Delhi', 'Guwahati', 'Hyderabad']
Total unique cities: 20


## 5. Handle Missing Values

Strategy per column (documented for transparency, as a real analyst report
should justify each choice):

| Column | Missing Reason (assumed) | Strategy |
|---|---|---|
| `Customer_Name` | Lost in export | Fill with `"Unknown Customer"` (Customer_ID still uniquely identifies them) |
| `Age` | Not collected at checkout | Impute with **median age** (robust to outliers) |
| `City` | Lost in export | Fill with `"Unknown"` — keep Region/State for geo-analysis |
| `Discount` | No discount applied | Fill with `0` (most defensible default for a missing discount) |


In [9]:
df['Customer_Name'] = df['Customer_Name'].fillna('Unknown Customer')

median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)
df['Age'] = df['Age'].astype(int)

df['City'] = df['City'].fillna('Unknown')

df['Discount'] = df['Discount'].fillna(0)

print("Remaining missing values:")
print(df.isna().sum())


Remaining missing values:
Order_ID            0
Order_Date          0
Customer_ID         0
Customer_Name       0
Gender              0
Age                 0
City                0
State               0
Region              0
Product_Category    0
Product_Name        0
Quantity            0
Unit_Price          0
Discount            0
Sales               0
Profit              0
Payment_Method      0
dtype: int64


## 6. Validate Numerical Columns

Check for impossible values: negative quantities, zero/negative prices,
discount outside [0, 1], and Sales/Profit consistency.


In [10]:
print("Quantity <= 0:", (df['Quantity'] <= 0).sum())
print("Unit_Price <= 0:", (df['Unit_Price'] <= 0).sum())
print("Discount outside [0,1]:", (~df['Discount'].between(0, 1)).sum())
print("Sales <= 0:", (df['Sales'] <= 0).sum())


Quantity <= 0: 37
Unit_Price <= 0: 24
Discount outside [0,1]: 0
Sales <= 0: 0


In [11]:
# Fix negative quantities (data entry sign error) -> take absolute value
df.loc[df['Quantity'] < 0, 'Quantity'] = df.loc[df['Quantity'] < 0, 'Quantity'].abs()

# Drop rows where Unit_Price is 0 (can't be a real transaction) — small count, safe to drop
zero_price_rows = (df['Unit_Price'] <= 0).sum()
df = df[df['Unit_Price'] > 0].reset_index(drop=True)
print(f"Dropped {zero_price_rows} rows with zero/negative Unit_Price")

# Recompute Sales for rows where Quantity sign was fixed, to keep Sales consistent
df['Sales'] = (df['Unit_Price'] * df['Quantity'] * (1 - df['Discount'])).round(2)

print("Final validation:")
print("Quantity <= 0:", (df['Quantity'] <= 0).sum())
print("Unit_Price <= 0:", (df['Unit_Price'] <= 0).sum())
print("Sales <= 0:", (df['Sales'] <= 0).sum())


Dropped 24 rows with zero/negative Unit_Price
Final validation:
Quantity <= 0: 0
Unit_Price <= 0: 0
Sales <= 0: 0


In [12]:
# Outlier check on Sales using IQR (informational — we keep them, e-commerce
# legitimately has high-value orders, but we flag for awareness)
Q1, Q3 = df['Sales'].quantile([0.25, 0.75])
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
outlier_count = (df['Sales'] > upper_bound).sum()
print(f"Sales values above IQR upper bound ({upper_bound:.2f}): {outlier_count} "
      f"({outlier_count/len(df)*100:.1f}% of rows) — retained as legitimate high-value orders")


Sales values above IQR upper bound (20139.63): 1652 (13.6% of rows) — retained as legitimate high-value orders


## 7. Final Checks & Save

In [13]:
print("Final shape:", df.shape)
print()
print(df.dtypes)
print()
df.describe(include='all').T.head(20)


Final shape: (12118, 17)

Order_ID                       str
Order_Date          datetime64[us]
Customer_ID                    str
Customer_Name               string
Gender                      string
Age                          int64
City                        string
State                       string
Region                      string
Product_Category            string
Product_Name                string
Quantity                     int64
Unit_Price                 float64
Discount                   float64
Sales                      float64
Profit                     float64
Payment_Method              string
dtype: object



,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Order_ID,12118,11978,ORD005513,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order_Date,12118,NaN,NaN,NaN,2023-09-14 20:59:15.438191,2022-01-01 00:00:00,2023-01-03 06:00:00,2023-10-16 00:00:00,2024-06-10 00:00:00,2024-12-28 00:00:00,NaN
Customer_ID,12118,1137,CUST00407,222,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer_Name,12118,1135,Lajita Walia,219,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,12118,2,Female,6417,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,12118.0,NaN,NaN,NaN,35.069814,18.0,29.0,35.0,40.0,70.0,9.1355
City,12118,21,Ahmedabad,881,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,12118,16,Maharashtra,1595,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Region,12118,4,West,3123,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product_Category,12118,7,Groceries,1809,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df.to_csv(CLEAN_PATH, index=False)
print(f"Saved cleaned dataset to {CLEAN_PATH}")
print(f"Rows: {len(df)} | Columns: {len(df.columns)}")


Saved cleaned dataset to ../data/ecommerce_clean.csv
Rows: 12118 | Columns: 17


---
**Next notebook:** `02_feature_engineering.ipynb` — derive Month/Quarter/Year,
Customer Lifetime Value, Average Order Value, Profit Margin, and Revenue per
Customer on top of this cleaned dataset.
